In [1]:
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
import os

In [2]:
with open('data.txt','r+') as file:
    data = file.read()
print(data)

[
  {
    "product_id": "MOB001",
    "brand": "Samsung",
    "model": "Galaxy S24 Ultra",
    "price": 129999.99,
    "release_year": 2024,
    "specs": {
      "ram": "12GB",
      "storage": "512GB",
      "battery": "5000mAh",
      "display": "6.8 inch AMOLED",
      "processor": "Snapdragon 8 Gen 3"
    },
    "ratings": {
      "average": 4.7,
      "count": 1250
    },
    "in_stock": true,
    "colors_available": ["Black", "Titanium Gray", "Violet"],
    "discount_percent": 10
  },
  {
    "product_id": "MOB002",
    "brand": "Apple",
    "model": "iPhone 15 Pro Max",
    "price": 159999.00,
    "release_year": 2023,
    "specs": {
      "ram": "8GB",
      "storage": "256GB",
      "battery": "4441mAh",
      "display": "6.7 inch Super Retina XDR",
      "processor": "A17 Pro"
    },
    "ratings": {
      "average": 4.8,
      "count": 2120
    },
    "in_stock": false,
    "colors_available": ["Blue Titanium", "White Titanium"],
    "discount_percent": 5
  },
  {
    "produ

In [3]:
template = ''' 
You are a helful assistant that can answer questions about the product.
Answer the question based on the product details provided.
product : {product}
question : {question}
'''

In [4]:
prompt_template = PromptTemplate.from_template(template)
prompt_template

PromptTemplate(input_variables=['product', 'question'], input_types={}, partial_variables={}, template=' \nYou are a helful assistant that can answer questions about the product.\nAnswer the question based on the product details provided.\nproduct : {product}\nquestion : {question}\n')

In [5]:
class Product(BaseModel):
    name:str =Field(description="Name of the product")
    price :str =Field(description="Price of the product")
    description : str = Field(description="Description of the product")
    category : str = Field(description="Category of the product")
    

In [6]:
gemini = ChatGoogleGenerativeAI(
    api_key = os.environ['gemini_api'],
    model = 'gemini-3.8-flash',
    temperature = 2.0
).with_structured_output(Product)
ollama = ChatOllama(
    model='qwen2.5:1.5b',
    temperature=0.2
).with_structured_output(Product)

In [7]:
chain = prompt_template | ollama

In [8]:
chain2 =prompt_template | gemini

In [9]:
user_input = [
    {
        "product" : data,
        "question" : "What is the price of iPhone 15 Pro Max?"
    },
    {
        "product" : data,
        "question" : "What is the price of Galaxy S24 Ultra?"
    },
    {
        "product" : data,
        "question" : "What is the price of Redmi?"
    },
    {
        "product" : data,
        "question" : "What is the price of OnePlus?"
    }
]

In [ ]:
u_input ={
        "product" : data,
        "question" : "What is the price of Redmi?"
    }
response = chain2.batch(user_input)     # Batch to invoke multiple prompt at once
response

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


GoogleAPIError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [ ]:
# u_input ={
#         "product" : data,
#         "question" : "What is the price of Smart Watch?"
#     }
# response = chain2.batch(user_input)
# response

In [ ]:
print(type(response))

In [ ]:
# response_dict = response.model_dump_json()
# print(response_dict)

In [ ]:
output =[]
for details in response:
    print(details.model_dump_json())


In [ ]:
results =[]
for query in user_input:
    print(chain.invoke(query).model_dump())